# Panel municipal 1/2: armado

Junta la deforestación del proyecto (IDEAM y GFW) con las tablas que compartió el profesor en BigQuery, y deja
el panel en `datos/analisis/`:

- `municipio_anio.parquet`: municipio × año.
- `municipio_mes.parquet`: municipio × mes.

El análisis está en `panel_municipal_analisis.ipynb`.

## 1. Descarga de las tablas del profesor

Baja cada tabla a `datos/profesor/<nombre>/` con `list_rows`, sin ejecutar SQL.
Si la tabla no cambió desde la última descarga, no la vuelve a bajar.

In [1]:
import json
import os
import shutil
import sys
from pathlib import Path

RAIZ = Path.cwd().resolve()
while not (RAIZ / "main_local.py").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.api_core import exceptions as gexc
from google.cloud import bigquery

from pipeline.config_local import DIR_DATOS, DIR_PANEL
from pipeline.entendimiento.describir_fuentes import _credenciales

credencial = _credenciales()
assert credencial, "Falta la llave en datos/logs/bigquery-key.json"
client = bigquery.Client(project="ofr-credito-deforestacion", credentials=credencial)

DIR_PROFESOR = DIR_DATOS / "profesor"
DIR_PROFESOR.mkdir(exist_ok=True)

In [2]:
TABLAS = {
    "finagro":      "prueba-ofr.Inclusion_Financiera.FINAGRO_Desembolsos_EFECTIVA",
    "sfc414":       "prueba-ofr.Inclusion_Financiera.SFC414_DASH",
    "municipios":   "prueba-ofr.Municipios.CODIGO",
    "eva":          "prueba-ofr.Produccion.eva_full_enriquecido",
    "ica_pecuario": "prueba-ofr.Produccion.ICA_pecuario",
}

In [3]:
def descargar(nombre, tabla, filas_por_parte=500_000):
    try:
        t = client.get_table(tabla)
    except (gexc.Forbidden, gexc.NotFound) as e:
        print(f"{nombre}: sin acceso ({type(e).__name__})")
        return
    if t.table_type != "TABLE":
        print(f"{nombre}: es {t.table_type}, se omite")
        return

    version = {"modificada": t.modified.isoformat(), "filas": t.num_rows}
    meta = DIR_PROFESOR / nombre / "_metadatos.json"
    if meta.exists() and json.loads(meta.read_text()) == version:
        print(f"{nombre}: al dia ({t.num_rows:,} filas)")
        return

    # Se baja a una carpeta temporal y solo reemplaza a la anterior si llega completa.
    temporal = DIR_PROFESOR / f"{nombre}.parcial"
    shutil.rmtree(temporal, ignore_errors=True)
    temporal.mkdir()
    lotes, filas, partes = [], 0, 0

    def escribir():
        nonlocal partes
        pq.write_table(pa.Table.from_batches(lotes), temporal / f"parte-{partes:05d}.parquet")
        partes += 1
        lotes.clear()

    for lote in client.list_rows(t, page_size=100_000).to_arrow_iterable():
        lotes.append(lote)
        filas += lote.num_rows
        if sum(b.num_rows for b in lotes) >= filas_por_parte:
            escribir()
            print(f"  {nombre}: {filas:,} / {t.num_rows:,}")
    if lotes:
        escribir()

    if filas != t.num_rows:
        print(f"{nombre}: INCOMPLETA ({filas:,} de {t.num_rows:,} filas), se reintenta en la proxima corrida")
        return
    (temporal / "_metadatos.json").write_text(json.dumps(version))
    shutil.rmtree(DIR_PROFESOR / nombre, ignore_errors=True)
    temporal.rename(DIR_PROFESOR / nombre)
    print(f"{nombre}: {filas:,} filas")


for nombre, tabla in TABLAS.items():
    descargar(nombre, tabla)

# consultar_tablas_profesor.ipynb lee los municipios de este CSV
pd.read_parquet(DIR_PROFESOR / "municipios").to_csv(DIR_PROFESOR / "CODIGO.csv", index=False)

finagro: al dia (6,158,859 filas)


sfc414: al dia (1,977,282 filas)


municipios: al dia (20,196 filas)


eva: al dia (347,141 filas)


ica_pecuario: al dia (8,976 filas)


## 2. Las tablas

In [4]:
def cargar(nombre):
    carpeta = DIR_PROFESOR / nombre
    esquema = pq.read_schema(next(carpeta.glob("parte-*.parquet")))
    texto = [c.name for c in esquema if pa.types.is_string(c.type)]
    # Texto como categorias (ocupa mucha menos memoria) y fechas como datetime
    tabla = pq.read_table(carpeta, read_dictionary=texto)
    return tabla.unify_dictionaries().to_pandas(date_as_object=False)


tablas = {nombre: cargar(nombre) for nombre in TABLAS}

### finagro

In [5]:
tablas["finagro"].info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6158859 entries, 0 to 6158858
Data columns (total 37 columns):
 #   Column                        Non-Null Count    Dtype         
---  ------                        --------------    -----         
 0   id_credito                    6158859 non-null  category      
 1   fecha_credito                 6158859 non-null  datetime64[us]
 2   cod_municipio                 6158859 non-null  category      
 3   tipo_cartera                  6158859 non-null  category      
 4   tipo_intermediario            6158859 non-null  category      
 5   tipo_productor                6158859 non-null  category      
 6   tipo_beneficiario             6158859 non-null  category      
 7   tipo_persona                  6158859 non-null  category      
 8   sexo                          6158859 non-null  category      
 9   cod_destino                   6158859 non-null  int64         
 10  linea_credito                 6158859 non-null  category      
 11

### sfc414

In [6]:
tablas["sfc414"].info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1977282 entries, 0 to 1977281
Data columns (total 17 columns):
 #   Column                               Non-Null Count    Dtype         
---  ------                               --------------    -----         
 0   Fecha_Corte                          1977282 non-null  datetime64[ms]
 1   Tipo_de_persona                      1977282 non-null  category      
 2   Sexo                                 1977282 non-null  category      
 3   Tipo_de_credito                      1977282 non-null  category      
 4   Tipo_de_garantia                     1977282 non-null  category      
 5   Montos_desembolsados                 1977282 non-null  float64       
 6   Numero_de_creditos_desembolsados     1977282 non-null  int64         
 7   Codigo_Municipio                     1977282 non-null  category      
 8   nombre_departamento                  1977065 non-null  category      
 9   nombre_municipio                     1977065 non-null  ca

### municipios

In [7]:
tablas["municipios"].info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20196 entries, 0 to 20195
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   cod_dane             20196 non-null  category
 1   nombre_departamento  20196 non-null  category
 2   nombre_municipio     20196 non-null  category
 3   region               20196 non-null  category
 4   fecha                20196 non-null  int64   
 5   ruralidad            20191 non-null  category
dtypes: category(5), int64(1)
memory usage: 379.2 KB


### eva

In [8]:
tablas["eva"].info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 347141 entries, 0 to 347140
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Cod_dep               347141 non-null  category      
 1   Departamento          347141 non-null  category      
 2   Cod_mun               347141 non-null  category      
 3   Municipio             347140 non-null  category      
 4   Grupo_cultivo         347141 non-null  category      
 5   Subgrupo              347141 non-null  category      
 6   Cultivo               347141 non-null  category      
 7   Desagregacion         347141 non-null  category      
 8   Year                  347141 non-null  datetime64[ms]
 9   Periodo               347141 non-null  category      
 10  Sembrada              347141 non-null  float64       
 11  Cosechada             347141 non-null  float64       
 12  Produccion            347141 non-null  float64       
 13 

### ica_pecuario

In [9]:
tablas["ica_pecuario"].info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8976 entries, 0 to 8975
Data columns (total 142 columns):
 #    Column                                                              Non-Null Count  Dtype   
---   ------                                                              --------------  -----   
 0    anio                                                                8976 non-null   int64   
 1    codigo_municipio                                                    8976 non-null   category
 2    departamento                                                        8976 non-null   category
 3    municipio                                                           8976 non-null   category
 4    total_aves_capacidad_instalada_engorde                              8976 non-null   float64 
 5    total_aves_capacidad_ocupada_engorde                                8976 non-null   float64 
 6    aves_no_de_predios_engorde                                          8976 non-null   float64 
 

## 3. Municipio × año

Una fila por municipio y año, 2020–2025: deforestación del IDEAM, alertas de GFW, crédito de FINAGRO, cultivos de
la EVA y ganado del ICA. Crédito, cultivos y ganado van también rezagados un año (`_t1`).
Se guarda en `datos/analisis/municipio_anio.parquet`.

In [10]:
import re
import unicodedata

DIR_ANALISIS = DIR_DATOS / "analisis"
DIR_ANALISIS.mkdir(exist_ok=True)


def normalizar(s):
    """Minusculas y sin tildes: 'Ganadería carne' y 'Ganaderia Leche' quedan comparables."""
    quitar = lambda x: unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().lower().strip()
    return s.map({v: quitar(v) for v in s.dropna().unique()})


# --- FINAGRO: credito en millones de pesos, por grupo de cadena
GRUPOS = {"ganaderia": r"^ganad", "palma": r"^palma de aceite$", "cacao": r"^cacao$",
          "cafe": r"^cafe$", "arroz": r"^arroz$"}
fin = tablas["finagro"][["fecha_credito", "cod_municipio", "valor_credito", "cadena", "linea_credito",
                         "tipo_productor", "tipo_persona"]].copy()
fin["cod_dane"] = fin.cod_municipio.astype(str)
fin["valor_mm"] = fin.valor_credito / 1e6
fin["anio"] = fin.fecha_credito.dt.year
# El municipio de los creditos grandes y de empresas suele ser el de su sede (Bogota, Medellin...).
# El de pequenos y medianos productores, persona natural, refleja mejor donde se usa la plata.
fin["pequeno_mediano"] = (fin.tipo_persona == "Natural") & fin.tipo_productor.isin(
    ["Pequeño", "Pequeño ppib", "Pequeño ingresos bajos", "Mediano"])
cadenas = normalizar(fin.cadena)
fin["grupo"] = cadenas.map({c: next((g for g, p in GRUPOS.items() if re.search(p, c)), "otros")
                            for c in cadenas.dropna().unique()})


def credito(df, tiempo):
    llave = ["cod_dane", tiempo]
    g = df.groupby(llave)
    out = pd.DataFrame({"credito_mm": g.valor_mm.sum(), "n_creditos": g.size(),
                        "credito_inversion_mm": df[df.linea_credito == "Inversión"].groupby(llave).valor_mm.sum()})
    pm = df[df.pequeno_mediano]
    out["credito_pequeno_mediano_mm"] = pm.groupby(llave).valor_mm.sum()
    out["n_creditos_pequeno_mediano"] = pm.groupby(llave).size()
    out["credito_ganaderia_pequeno_mediano_mm"] = pm[pm.grupo == "ganaderia"].groupby(llave).valor_mm.sum()
    por_grupo = df.groupby(llave + ["grupo"]).valor_mm.sum().unstack()
    return out.join(por_grupo.drop(columns="otros").add_prefix("credito_").add_suffix("_mm"))


# --- EVA: hectareas sembradas y cosechadas (los dos semestres se suman en el anio)
eva = tablas["eva"][["Cod_mun", "Year", "Cultivo", "Sembrada", "Cosechada"]].copy()
eva["cod_dane"] = eva.Cod_mun.astype(str).str.zfill(5)  # 2006-2018 trae Antioquia sin el cero inicial
eva["anio"] = eva.Year.dt.year
eva["cultivo"] = normalizar(eva.Cultivo)

# --- ICA: inventario bovino
ica = tablas["ica_pecuario"][["codigo_municipio", "anio", "total_bovinos", "total_fincas_con_bovinos"]].copy()
ica["cod_dane"] = ica.codigo_municipio.astype(str)

# --- Deforestacion: los paneles vienen por celda de 5 km; se suman por municipio
leer = lambda t, cols: pd.read_parquet(DIR_PANEL / f"{t}.parquet", columns=cols)
ideam = leer("panel_ideam", ["cod_dane", "cell_id", "periodo", "def_ha", "bosque_ideam_ha", "sin_info_ha"])
ideam["anio"] = ideam.periodo.astype(str).str[-4:].astype(int)  # el periodo 2020-2021 es el anio 2021
gfw = leer("panel_deforestacion_colombia", ["cod_dane", "periodo", "anio", "area_def_ha"])
gfw["periodo"] = gfw.periodo.astype("datetime64[ns]")

# --- Atributos del municipio: el registro mas reciente de la tabla del profesor
municipios = (tablas["municipios"].assign(cod_dane=lambda d: d.cod_dane.astype(str))
              .sort_values("fecha").groupby("cod_dane")
              [["nombre_departamento", "nombre_municipio", "region", "ruralidad"]].last())
# Bosque del IDEAM al inicio del primer periodo (2021) y area de la grilla de cada municipio
primer_anio = ideam[ideam.anio == ideam.anio.min()]
fijas = primer_anio.groupby("cod_dane").agg(bosque_ideam_inicial_ha=("bosque_ideam_ha", "sum"),
                                            n_celdas=("cell_id", "size"))
fijas["area_grilla_ha"] = fijas.n_celdas * 2500  # celdas de 5 x 5 km

# 27493 (Belen de Bajira, Choco) esta en los paneles y en la EVA, pero no en la lista del profesor
codigos = municipios.index.union(fijas.index)

In [11]:
ANIOS = range(2019, 2026)  # 2019 solo sirve para el rezago de 2020; 2026 va incompleto
llave = ["cod_dane", "anio"]

deforestacion = pd.concat([
    ideam.groupby(llave)[["def_ha", "bosque_ideam_ha", "sin_info_ha"]].sum()
         .rename(columns={"def_ha": "def_ideam_ha", "sin_info_ha": "sin_info_ideam_ha"}),
    gfw.groupby(llave).area_def_ha.sum().rename("alertas_gfw_ha"),
], axis=1)

agricola = eva.groupby(llave).agg(area_sembrada_ha=("Sembrada", "sum"), area_cosechada_ha=("Cosechada", "sum"))
for cultivo, nombre in {"palma de aceite": "palma", "cacao": "cacao", "cafe": "cafe", "arroz": "arroz"}.items():
    agricola[f"sembrada_{nombre}_ha"] = eva[eva.cultivo == cultivo].groupby(llave).Sembrada.sum()

pecuario = ica.groupby(llave)[["total_bovinos", "total_fincas_con_bovinos"]].sum().rename(
    columns={"total_bovinos": "bovinos", "total_fincas_con_bovinos": "fincas_bovinas"})

esqueleto = pd.MultiIndex.from_product([codigos, ANIOS], names=llave)
anual = (pd.DataFrame(index=esqueleto).join(deforestacion).join(credito(fin, "anio")).join(agricola).join(pecuario))

# Sin creditos o sin cultivos reportados es cero, dentro de los anios que cubre cada fuente
col_credito = [c for c in anual if c.startswith(("credito_", "n_creditos"))]
col_agricola = list(agricola.columns)
anual[col_credito] = anual[col_credito].fillna(0)
cubre_eva = anual.index.get_level_values("anio") <= 2024
anual.loc[cubre_eva, col_agricola] = anual.loc[cubre_eva, col_agricola].fillna(0)

# Rezago de un anio para las explicativas: credito, cultivos y ganado de t-1
explicativas = col_credito + col_agricola + list(pecuario.columns)
anual = anual.join(anual.groupby(level="cod_dane")[explicativas].shift(1).add_suffix("_t1"))

anual = anual.query("anio >= 2020").join(fijas).join(municipios).reset_index()
anual["tasa_def_ideam"] = anual.def_ideam_ha / anual.bosque_ideam_ha
anual.to_parquet(DIR_ANALISIS / "municipio_anio.parquet", index=False)

# Chequeo: que parte de cada fuente quedo en la tabla (2020-2025)
en = lambda df, col, anio="anio": df.loc[df[anio].between(2020, 2025), col].sum()
pd.DataFrame({"en la fuente": [en(ideam, "def_ha"), en(fin, "valor_mm"),
                               en(eva, "Sembrada"), en(ica, "total_bovinos")],
              "en la tabla": [anual.def_ideam_ha.sum(), anual.credito_mm.sum(),
                              anual.area_sembrada_ha.sum(), anual.bovinos.sum()]},
             index=["IDEAM def_ha", "FINAGRO millones", "EVA ha sembradas", "ICA bovinos"]
             ).assign(porcentaje=lambda d: (100 * d["en la tabla"] / d["en la fuente"]).round(2))

,en la fuente,en la tabla,porcentaje
IDEAM def_ha,6.085546e+05,6.085546e+05,100.0
FINAGRO millones,1.925571e+08,1.925571e+08,100.0
EVA ha sembradas,2.722722e+07,2.722722e+07,100.0
ICA bovinos,1.747100e+08,1.747100e+08,100.0


In [12]:
print(anual.shape)
anual.head()

(6738, 52)


,cod_dane,anio,def_ideam_ha,bosque_ideam_ha,sin_info_ideam_ha,alertas_gfw_ha,credito_mm,n_creditos,credito_inversion_mm,credito_pequeno_mediano_mm,...,bovinos_t1,fincas_bovinas_t1,bosque_ideam_inicial_ha,n_celdas,area_grilla_ha,nombre_departamento,nombre_municipio,region,ruralidad,tasa_def_ideam
0,05001,2020,NaN,NaN,NaN,10.57346,1.921468e+06,2105.0,146305.685430,19817.336405,...,13462.0,692.0,7782.125,16.0,40000.0,Antioquia,Medellín,Eje cafetero,Ciudades y aglomeraciones,NaN
1,05001,2021,1.6875,7782.1250,0.0,31.36564,2.431347e+06,1786.0,232787.652929,18334.179579,...,13317.0,770.0,7782.125,16.0,40000.0,Antioquia,Medellín,Eje cafetero,Ciudades y aglomeraciones,0.000217
2,05001,2022,0.0000,7764.1875,0.0,17.39001,2.593599e+06,2525.0,243660.293639,31891.863193,...,13373.0,635.0,7782.125,16.0,40000.0,Antioquia,Medellín,Eje cafetero,Ciudades y aglomeraciones,0.000000
3,05001,2023,1.8750,7686.1250,0.0,47.59305,1.573201e+06,2807.0,424626.012285,22887.317339,...,13787.0,582.0,7782.125,16.0,40000.0,Antioquia,Medellín,Eje cafetero,Ciudades y aglomeraciones,0.000244
4,05001,2024,16.5625,7684.2500,0.0,75.03028,3.100697e+06,3480.0,292669.511618,16515.752303,...,12934.0,583.0,7782.125,16.0,40000.0,Antioquia,Medellín,Eje cafetero,Ciudades y aglomeraciones,0.002155


## 4. Municipio × mes

Alertas de GFW y crédito de FINAGRO, enero 2020 – junio 2026. Se guarda en `datos/analisis/municipio_mes.parquet`.

In [13]:
MESES = pd.date_range("2020-01-01", "2026-06-01", freq="MS")  # donde GFW y FINAGRO estan completos
fin["mes"] = fin.fecha_credito.dt.to_period("M").dt.to_timestamp()

alertas = gfw.groupby(["cod_dane", "periodo"]).area_def_ha.sum().rename("alertas_gfw_ha")
alertas.index = alertas.index.set_names(["cod_dane", "mes"])

esqueleto = pd.MultiIndex.from_product([codigos, MESES], names=["cod_dane", "mes"])
mensual = pd.DataFrame(index=esqueleto).join(alertas).join(credito(fin[fin.mes.isin(MESES)], "mes"))
col_credito = [c for c in mensual if c.startswith(("credito_", "n_creditos"))]
mensual[col_credito] = mensual[col_credito].fillna(0)
mensual = mensual.join(municipios[["region"]]).reset_index()
mensual.to_parquet(DIR_ANALISIS / "municipio_mes.parquet", index=False)

# Chequeo: que parte de cada fuente quedo en la tabla
pd.DataFrame({"en la fuente": [gfw.loc[gfw.periodo.isin(MESES), "area_def_ha"].sum(),
                               fin.loc[fin.mes.isin(MESES), "valor_mm"].sum()],
              "en la tabla": [mensual.alertas_gfw_ha.sum(), mensual.credito_mm.sum()]},
             index=["GFW ha", "FINAGRO millones"]
             ).assign(porcentaje=lambda d: (100 * d["en la tabla"] / d["en la fuente"]).round(2))

,en la fuente,en la tabla,porcentaje
GFW ha,1.082380e+06,1.082380e+06,100.0
FINAGRO millones,2.182778e+08,2.182778e+08,100.0


In [14]:
print(mensual.shape)
mensual.head()

(87594, 15)


,cod_dane,mes,alertas_gfw_ha,credito_mm,n_creditos,credito_inversion_mm,credito_pequeno_mediano_mm,n_creditos_pequeno_mediano,credito_ganaderia_pequeno_mediano_mm,credito_arroz_mm,credito_cacao_mm,credito_cafe_mm,credito_ganaderia_mm,credito_palma_mm,region
0,05001,2020-01-01,1.19930,71389.277700,233.0,26733.388263,3508.441349,72.0,1060.742095,46.100000,0.0,1702.553410,18093.929462,0.000000,Eje cafetero
1,05001,2020-02-01,3.64687,228679.557371,382.0,29106.588239,2399.288435,64.0,571.051675,1844.358718,0.0,2556.659889,54928.720565,458.552188,Eje cafetero
2,05001,2020-03-01,0.79545,129621.640210,216.0,14270.037323,1781.971992,68.0,662.349960,39.720000,0.0,2985.981827,33319.150727,0.000000,Eje cafetero
3,05001,2020-04-01,0.22028,301960.237083,168.0,11061.776244,1699.310841,24.0,310.000000,467.000000,0.0,35.437800,10094.808973,0.000000,Eje cafetero
4,05001,2020-05-01,0.22028,71471.895861,208.0,8642.366446,2199.817757,23.0,274.817757,149.888442,0.0,2994.182055,23585.434715,0.000000,Eje cafetero


Siguiente: `panel_municipal_analisis.ipynb`, que analiza estas dos tablas.